# ModelLog Anatomy

Inspects the model-level container, field access, layer dictionaries, and summary-facing state.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerPassLog.func_config",
    "tl.LayerPassLog.func_is_inplace",
    "tl.LayerPassLog.func_kwargs_non_tensor",
    "tl.LayerPassLog.func_name",
    "tl.LayerPassLog.func_non_tensor_args",
    "tl.LayerPassLog.func_positional_args_non_tensor",
    "tl.LayerPassLog.func_rng_states",
    "tl.LayerPassLog.func_time",
    "tl.LayerPassLog.get_child_layers",
    "tl.LayerPassLog.get_parent_layers",
    "tl.LayerPassLog.grad_dtype",
    "tl.LayerPassLog.grad_fn_id",
    "tl.LayerPassLog.grad_fn_name",
    "tl.LayerPassLog.grad_fn_object",
    "tl.LayerPassLog.grad_memory",
    "tl.LayerPassLog.grad_memory_str",
    "tl.LayerPassLog.grad_shape",
    "tl.LayerPassLog.gradient",
    "tl.LayerPassLog.has_child_tensor_variations",
    "tl.LayerPassLog.has_children",
    "tl.LayerPassLog.has_co_parents",
    "tl.LayerPassLog.has_gradient",
    "tl.LayerPassLog.has_input_ancestor",
    "tl.LayerPassLog.has_internally_initialized_ancestor",
    "tl.LayerPassLog.has_parents",
    "tl.LayerPassLog.has_saved_activations",
    "tl.LayerPassLog.has_siblings",
    "tl.LayerPassLog.in_cond_branch",
    "tl.LayerPassLog.input_ancestors",
    "tl.LayerPassLog.internally_initialized_ancestors",
    "tl.LayerPassLog.internally_initialized_parents",
    "tl.LayerPassLog.intervention_log",
    "tl.LayerPassLog.io_role",
    "tl.LayerPassLog.is_buffer_layer",
    "tl.LayerPassLog.is_computed_inside_submodule",
    "tl.LayerPassLog.is_final_output",
    "tl.LayerPassLog.is_input_layer",
    "tl.LayerPassLog.is_internally_initialized",
    "tl.LayerPassLog.is_internally_terminated",
    "tl.LayerPassLog.is_leaf_module_output",
    "tl.LayerPassLog.is_output_ancestor",
    "tl.LayerPassLog.is_output_layer",
    "tl.LayerPassLog.is_part_of_iterable_output",
    "tl.LayerPassLog.is_scalar_bool",
    "tl.LayerPassLog.is_submodule_input",
    "tl.LayerPassLog.is_submodule_output",
    "tl.LayerPassLog.is_terminal_bool_layer",
    "tl.LayerPassLog.iterable_output_index",
    "tl.LayerPassLog.layer_label",
    "tl.LayerPassLog.layer_label_no_pass",
    "tl.LayerPassLog.layer_label_no_pass_short",
    "tl.LayerPassLog.layer_label_raw",
    "tl.LayerPassLog.layer_label_short",
    "tl.LayerPassLog.layer_label_w_pass",
    "tl.LayerPassLog.layer_label_w_pass_short",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")